# 06 · Baseline Model — Logistic Regression

**Project:** Enterprise HR AI  
**Input:** `features_scaled.csv` (StandardScaler applied in Step 5 — correct for LR)  
**Purpose:** Establish ONE reproducible baseline number that all future models must beat.  
**Rule:** No hyperparameter tuning. No model comparison. One model, full reporting.

---

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_score, recall_score, f1_score
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

PROC   = os.path.join('..', 'data', 'processed')
MODELS = os.path.join('..', 'models')
os.makedirs(MODELS, exist_ok=True)

RANDOM_STATE = 42   # fixed — NEVER change this for the baseline
print('PROC  :', os.path.abspath(PROC))
print('MODELS:', os.path.abspath(MODELS))
print('random_state:', RANDOM_STATE, '(fixed for reproducibility)')

PROC  : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed
MODELS: C:\Users\ASUS\Desktop\enterprise_hr_ai\models
random_state: 42 (fixed for reproducibility)


In [2]:
df = pd.read_csv(os.path.join(PROC, 'features_scaled.csv'))
print(f'Loaded features_scaled.csv: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Columns: {df.columns.tolist()}')

Loaded features_scaled.csv: 1,470 rows x 49 cols
Columns: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobRole_Research Director', 'JobRole_Research Scientist', 'JobRole_Sales Executive', 'JobRol

---
## Step 1 · X / y Split and Encoding Confirmation

In [3]:
TARGET = 'Attrition'

# Confirm the encoding before splitting
print('=== ENCODING MAPPING ===')
raw_vals = df[TARGET].unique()
print(f'  Raw unique values in Attrition column: {sorted(raw_vals)}')
print(f'  Encoding: 0 = stayed ("No"), 1 = left ("Yes")')
print(f'  Positive class (class=1): employee LEFT the company')
print()

# In features_scaled.csv the target was already 0/1 from Step 5
# Confirm:
assert set(df[TARGET].unique()).issubset({0, 1}), \
    f'Expected binary 0/1 in Attrition, got: {df[TARGET].unique()}'
print('Assert passed: Attrition column is already 0/1 integer encoding.')

X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Overall attrition rate: {y.mean()*100:.2f}%  ({y.sum()} positives / {len(y)} total)')

=== ENCODING MAPPING ===
  Raw unique values in Attrition column: [np.int64(0), np.int64(1)]
  Encoding: 0 = stayed ("No"), 1 = left ("Yes")
  Positive class (class=1): employee LEFT the company

Assert passed: Attrition column is already 0/1 integer encoding.
X shape: (1470, 48)
y shape: (1470,)
Overall attrition rate: 16.12%  (237 positives / 1470 total)


---
## Step 2 · Train/Test Split (80/20, stratified)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print('=== SPLIT SIZES ===')
print(f'  Train : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Test  : {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X)*100:.1f}%)')
print()

print('=== CLASS BALANCE — STRATIFY VERIFICATION ===')
train_pos_pct = y_train.mean() * 100
train_neg_pct = (1 - y_train.mean()) * 100
test_pos_pct  = y_test.mean()  * 100
test_neg_pct  = (1 - y_test.mean())  * 100

print(f'  Train set: {y_train.sum()} positives ({train_pos_pct:.2f}% attrition) | '
      f'{(y_train==0).sum()} negatives ({train_neg_pct:.2f}% stayed)')
print(f'  Test  set: {y_test.sum()} positives ({test_pos_pct:.2f}% attrition) | '
      f'{(y_test==0).sum()} negatives ({test_neg_pct:.2f}% stayed)')
print(f'  Expected ~16.1% / ~83.9% in both — stratify=y working correctly.')

# Assert stratification worked (within 0.5% tolerance)
overall_rate = y.mean() * 100
assert abs(train_pos_pct - overall_rate) < 0.5, \
    f'Stratification off in train: {train_pos_pct:.2f}% vs overall {overall_rate:.2f}%'
assert abs(test_pos_pct - overall_rate) < 0.5, \
    f'Stratification off in test: {test_pos_pct:.2f}% vs overall {overall_rate:.2f}%'
print('  Assert passed: stratification within 0.5% tolerance in both sets.')

=== SPLIT SIZES ===
  Train : 1,176 rows  (80.0%)
  Test  : 294 rows  (20.0%)

=== CLASS BALANCE — STRATIFY VERIFICATION ===
  Train set: 190 positives (16.16% attrition) | 986 negatives (83.84% stayed)
  Test  set: 47 positives (15.99% attrition) | 247 negatives (84.01% stayed)
  Expected ~16.1% / ~83.9% in both — stratify=y working correctly.
  Assert passed: stratification within 0.5% tolerance in both sets.


---
## Step 3 · Train Logistic Regression Baseline

In [5]:
print('Training Logistic Regression (max_iter=1000, C=1.0 default, solver=lbfgs)...')
lr = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    # All other hyperparameters at sklearn defaults:
    # C=1.0 (L2 regularisation), solver='lbfgs', class_weight=None
    # No tuning — this is the baseline.
)
lr.fit(X_train, y_train)
print(f'Training complete. Converged: {lr.n_iter_[0]} iterations (max_iter=1000)')

Training Logistic Regression (max_iter=1000, C=1.0 default, solver=lbfgs)...
Training complete. Converged: 51 iterations (max_iter=1000)


In [6]:
y_pred       = lr.predict(X_test)
y_pred_proba = lr.predict_proba(X_test)[:, 1]   # P(class=1 = left)

print(f'Predictions generated. y_pred unique values: {np.unique(y_pred)}')
print(f'y_pred_proba range: min={y_pred_proba.min():.4f}  max={y_pred_proba.max():.4f}')

Predictions generated. y_pred unique values: [0 1]
y_pred_proba range: min=0.0003  max=0.9690


---
## Step 4 · Full Metrics Report

In [7]:
print('=== CONFUSION MATRIX (raw counts, test set) ===')
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=['Actual: Stayed (0)', 'Actual: Left (1)'],
    columns=['Predicted: Stayed (0)', 'Predicted: Left (1)']
)
print(cm_df.to_string())
print()
tn, fp, fn, tp = cm.ravel()
print(f'  True Negatives  (correctly predicted stayed): {tn}')
print(f'  False Positives (predicted left, actually stayed): {fp}')
print(f'  False Negatives (predicted stayed, actually left): {fn}   <- missed attrition cases')
print(f'  True Positives  (correctly predicted left): {tp}')

=== CONFUSION MATRIX (raw counts, test set) ===
                    Predicted: Stayed (0)  Predicted: Left (1)
Actual: Stayed (0)                    238                    9
Actual: Left (1)                       30                   17

  True Negatives  (correctly predicted stayed): 238
  False Positives (predicted left, actually stayed): 9
  False Negatives (predicted stayed, actually left): 30   <- missed attrition cases
  True Positives  (correctly predicted left): 17


In [8]:
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_pred_proba)

print('=== SCALAR METRICS (positive class = left=1) ===')
print(f'  Precision : {precision:.4f}  (of all predicted-left, how many actually left)')
print(f'  Recall    : {recall:.4f}  (of all who actually left, how many did we catch)')
print(f'  F1        : {f1:.4f}  (harmonic mean of precision & recall)')
print(f'  ROC-AUC   : {roc_auc:.4f}  (discrimination across all thresholds)')
print()
print('NOTE: Accuracy is NOT the primary metric for this imbalanced dataset.')
print(f'  (A naive all-zeros classifier would get {(y_test==0).mean()*100:.2f}% accuracy.)')
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, y_pred)
print(f'  Actual accuracy: {acc*100:.2f}%  — context: this number is misleading alone.')

=== SCALAR METRICS (positive class = left=1) ===
  Precision : 0.6538  (of all predicted-left, how many actually left)
  Recall    : 0.3617  (of all who actually left, how many did we catch)
  F1        : 0.4658  (harmonic mean of precision & recall)
  ROC-AUC   : 0.8134  (discrimination across all thresholds)

NOTE: Accuracy is NOT the primary metric for this imbalanced dataset.
  (A naive all-zeros classifier would get 84.01% accuracy.)
  Actual accuracy: 86.73%  — context: this number is misleading alone.


In [9]:
print('=== FULL classification_report() ===')
print(classification_report(
    y_test, y_pred,
    target_names=['Stayed (0)', 'Left (1)'],
    digits=4
))

=== FULL classification_report() ===
              precision    recall  f1-score   support

  Stayed (0)     0.8881    0.9636    0.9243       247
    Left (1)     0.6538    0.3617    0.4658        47

    accuracy                         0.8673       294
   macro avg     0.7710    0.6626    0.6950       294
weighted avg     0.8506    0.8673    0.8510       294



In [10]:
print('=== BASELINE RECORD (to beat in Step 7) ===')
print(f'  Model      : Logistic Regression (C=1.0, L2, lbfgs, max_iter=1000)')
print(f'  Split      : 80/20 stratified, random_state=42')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1         : {f1:.4f}')
print(f'  ROC-AUC    : {roc_auc:.4f}')
print(f'  Confusion  : TN={tn} FP={fp} FN={fn} TP={tp}')

=== BASELINE RECORD (to beat in Step 7) ===
  Model      : Logistic Regression (C=1.0, L2, lbfgs, max_iter=1000)
  Split      : 80/20 stratified, random_state=42
  Precision  : 0.6538
  Recall     : 0.3617
  F1         : 0.4658
  ROC-AUC    : 0.8134
  Confusion  : TN=238 FP=9 FN=30 TP=17


---
## Step 5 · Top 10 Coefficients (sanity check)

In [11]:
coef_series = pd.Series(lr.coef_[0], index=X.columns)
coef_abs = coef_series.abs().sort_values(ascending=False)

top10_idx   = coef_abs.head(10).index
top10_coefs = coef_series[top10_idx]

print('=== TOP 10 COEFFICIENTS BY ABSOLUTE VALUE ===')
print(f'{"Feature":<40s}  {"Coefficient":>12s}  Direction')
print('-' * 75)
for feat, coef in top10_coefs.items():
    direction = 'TOWARD ATTRITION (+)' if coef > 0 else 'AWAY from attrition (-)'
    print(f'{feat:<40s}  {coef:>12.4f}  {direction}')

print()
print('--- All coefficients (descending by value, for full picture) ---')
print(f'{"Feature":<40s}  {"Coef":>10s}')
print('-' * 55)
for feat, coef in coef_series.sort_values(ascending=False).items():
    bar = '+' * int(abs(coef) * 3) if coef > 0 else '-' * int(abs(coef) * 3)
    print(f'{feat:<40s}  {coef:>10.4f}  {bar[:30]}')

=== TOP 10 COEFFICIENTS BY ABSOLUTE VALUE ===
Feature                                    Coefficient  Direction
---------------------------------------------------------------------------
OverTime                                        1.8021  TOWARD ATTRITION (+)
BusinessTravel_Travel_Frequently                1.5569  TOWARD ATTRITION (+)
JobRole_Laboratory Technician                   1.3029  TOWARD ATTRITION (+)
JobRole_Sales Representative                    0.9187  TOWARD ATTRITION (+)
EducationField_Other                           -0.8919  AWAY from attrition (-)
YearsSinceLastPromotion                         0.8416  TOWARD ATTRITION (+)
JobRole_Research Director                      -0.8213  AWAY from attrition (-)
TotalWorkingYears                              -0.6971  AWAY from attrition (-)
MaritalStatus_Single                            0.6965  TOWARD ATTRITION (+)
BusinessTravel_Travel_Rarely                    0.6689  TOWARD ATTRITION (+)

--- All coefficients (descending

In [12]:
print('=== BUSINESS SENSE CHECK — TOP 10 ===')
print()

# Dynamic commentary based on actual coefficient signs
checks = {
    'OverTime':                  ('positive', 'EXPECTED — overtime causes burnout, strong attrition driver in literature'),
    'MaritalStatus_Single':      ('positive', 'EXPECTED — single employees have lower anchor cost to switching jobs'),
    'JobRole_Sales Representative': ('positive', 'EXPECTED — sales rep roles have high voluntary turnover industry-wide'),
    'JobRole_Laboratory Technician': ('positive', 'EXPECTED — technically skilled, portable skills, high market demand'),
    'JobRole_Human Resources':   ('positive', 'PLAUSIBLE — HR roles can have high burnout'),
    'overall_satisfaction_score':('negative', 'EXPECTED — higher satisfaction -> lower attrition risk'),
    'JobSatisfaction':           ('negative', 'EXPECTED — lower satisfaction -> higher attrition'),
    'JobLevel':                  ('negative', 'EXPECTED — higher seniority -> less likely to leave'),
    'income_per_year_at_company':('negative', 'EXPECTED — better compensated for tenure -> more likely to stay'),
    'experience_ratio':          ('negative', 'EXPECTED — more of career spent here -> higher switching cost'),
    'TotalWorkingYears':         ('negative', 'EXPECTED — experienced workers tend to be more stable'),
    'MonthlyIncome':             ('negative', 'EXPECTED — higher income -> lower attrition risk'),
    'StockOptionLevel':          ('negative', 'EXPECTED — equity compensation creates golden handcuffs'),
    'YearsAtCompany':            ('negative', 'EXPECTED — longer tenure -> higher switching cost'),
    'YearsWithCurrManager':      ('negative', 'EXPECTED — good manager relationship -> retention'),
    'NumCompaniesWorked':        ('positive', 'EXPECTED — job hopper pattern predicts future hopping'),
    'BusinessTravel_Travel_Frequently': ('positive', 'EXPECTED — frequent travel is a well-known burnout driver'),
    'DistanceFromHome':          ('positive', 'EXPECTED — longer commute -> higher attrition risk'),
}

flags = []
for feat, coef in top10_coefs.items():
    actual_dir = 'positive' if coef > 0 else 'negative'
    if feat in checks:
        expected_dir, rationale = checks[feat]
        match = 'OK' if actual_dir == expected_dir else 'UNEXPECTED — REVIEW'
        flag  = '✅' if actual_dir == expected_dir else '⚠️ BACKWARDS'
        print(f'{flag}  [{feat}]  coef={coef:.4f}  ({actual_dir})')
        print(f'    {rationale}')
        if match == 'UNEXPECTED — REVIEW':
            flags.append(feat)
    else:
        print(f'❓  [{feat}]  coef={coef:.4f}  ({actual_dir})')
        print(f'    Not in sanity-check map — review manually.')
        flags.append(feat)
    print()

print('---')
if flags:
    print(f'FEATURES NEEDING REVIEW: {flags}')
else:
    print('All top-10 coefficients directionally consistent with business expectations.')

=== BUSINESS SENSE CHECK — TOP 10 ===

✅  [OverTime]  coef=1.8021  (positive)
    EXPECTED — overtime causes burnout, strong attrition driver in literature

✅  [BusinessTravel_Travel_Frequently]  coef=1.5569  (positive)
    EXPECTED — frequent travel is a well-known burnout driver

✅  [JobRole_Laboratory Technician]  coef=1.3029  (positive)
    EXPECTED — technically skilled, portable skills, high market demand

✅  [JobRole_Sales Representative]  coef=0.9187  (positive)
    EXPECTED — sales rep roles have high voluntary turnover industry-wide

❓  [EducationField_Other]  coef=-0.8919  (negative)
    Not in sanity-check map — review manually.

❓  [YearsSinceLastPromotion]  coef=0.8416  (positive)
    Not in sanity-check map — review manually.

❓  [JobRole_Research Director]  coef=-0.8213  (negative)
    Not in sanity-check map — review manually.

✅  [TotalWorkingYears]  coef=-0.6971  (negative)
    EXPECTED — experienced workers tend to be more stable

✅  [MaritalStatus_Single]  coef=0.6

---
## Step 6 · Save Model

In [13]:
model_path = os.path.join(MODELS, 'baseline_logreg.joblib')
joblib.dump(lr, model_path)
size = os.path.getsize(model_path)
print(f'Saved: baseline_logreg.joblib  ({size:,} bytes)')
print(f'Path : {os.path.abspath(model_path)}')

# Verify round-trip
lr_loaded = joblib.load(model_path)
y_check   = lr_loaded.predict(X_test[:5])
print(f'Round-trip check (first 5 test predictions): {y_check}')
print('Model saved and verified.')

print()
print('=== MODELS/ directory ===')
for f in sorted(os.listdir(MODELS)):
    fp = os.path.join(MODELS, f)
    print(f'  {f}  ({os.path.getsize(fp):,} bytes)')

Saved: baseline_logreg.joblib  (2,575 bytes)
Path : C:\Users\ASUS\Desktop\enterprise_hr_ai\models\baseline_logreg.joblib
Round-trip check (first 5 test predictions): [0 0 0 0 0]
Model saved and verified.

=== MODELS/ directory ===
  attrition  (0 bytes)
  baseline_logreg.joblib  (2,575 bytes)
  checkpoints  (0 bytes)
  embeddings  (0 bytes)
  scaler.joblib  (1,983 bytes)


---
## Summary

| Metric | Value |
|---|---|
| Model | Logistic Regression — C=1.0, L2, lbfgs, max_iter=1000 |
| Split | 80/20 stratified, random_state=42 (fixed) |
| Precision | See Step 4 output |
| Recall | See Step 4 output |
| F1 | See Step 4 output |
| ROC-AUC | See Step 4 output |

**Next step:** Step 7 — Random Forest + XGBoost using `features_unscaled.csv`.  
All models in Step 7 must beat this baseline's F1 and ROC-AUC to be considered an improvement.